# SuperKart Sales Forecasting — End-to-End MLOps Pipeline with CI/CD

| Deliverable | Link |
|---|---|
| **GitHub repository** | [https://github.com/shravanambudkar/superkart-mlops](https://github.com/shravanambudkar/superkart-mlops) |
| **GitHub Actions runs** | [https://github.com/shravanambudkar/superkart-mlops/actions](https://github.com/shravanambudkar/superkart-mlops/actions) |
| **Hugging Face Space (Streamlit frontend)** | [https://huggingface.co/spaces/shravanda/SuperKart-Sales-Forecast](https://huggingface.co/spaces/shravanda/SuperKart-Sales-Forecast) |
| Hugging Face Dataset | [https://huggingface.co/datasets/shravanda/superkart-sales-data](https://huggingface.co/datasets/shravanda/superkart-sales-data) |
| Hugging Face Model | [https://huggingface.co/shravanda/superkart-sales-model](https://huggingface.co/shravanda/superkart-sales-model) |

## Business Context
A sales forecast predicts future revenue from historical data, industry trends and the current pipeline. SuperKart uses forecasts to plan
sales operations by region, guide supply-chain procurement, reduce pipeline/forecast risk and set benchmarks for future trend analysis.

## Objective
As the MLOps Engineer, build an **automated, reproducible pipeline** that:
1. **Registers** the raw data on the Hugging Face Hub (dataset repo),
2. **Prepares** it (cleaning, feature selection, train/test split) and pushes the splits back to the Hub,
3. **Trains, tunes and evaluates** tree-based regressors and **registers** the best model in the Hugging Face model hub,
4. **Deploys** a Streamlit app (Docker) to a Hugging Face Space that serves predictions of `Product_Store_Sales_Total`,
5. **Automates** all of the above with a **GitHub Actions** CI/CD workflow that runs on every push to `main`.

## Data Dictionary
| Column | Description |
|---|---|
| Product_Id | Unique product identifier (2-letter prefix + number) |
| Product_Weight | Weight of the product |
| Product_Sugar_Content | Low Sugar / Regular / No Sugar |
| Product_Allocated_Area | Share of the store's display area allocated to the product |
| Product_Type | Broad product category (Dairy, Snack Foods, Meat, …) |
| Product_MRP | Maximum retail price |
| Store_Id | Unique store identifier |
| Store_Establishment_Year | Year the store was established |
| Store_Size | High / Medium / Small |
| Store_Location_City_Type | Tier 1 / Tier 2 / Tier 3 city |
| Store_Type | Departmental Store, Supermarket Type 1/2, Food Mart |
| **Product_Store_Sales_Total** | **Target** — total revenue of the product in that store |

## Pipeline architecture
```
 SuperKart.csv ──► data_register.py ──► HF Dataset repo (raw)
                                             │
                    prep.py ◄────────────────┘   clean ▸ drop IDs ▸ split 80/20
                       │
                       └──► HF Dataset repo (Xtrain / Xtest / ytrain / ytest)
                                             │
                    train.py ◄───────────────┘   GridSearchCV (RandomForest, XGBoost) ▸ evaluate
                       │
                       └──► HF Model hub (best_superkart_model_v1.joblib + metrics)
                                             │
                    hosting.py ──► HF Space (Docker + Streamlit app.py loads model from the hub)

 GitHub Actions (pipeline.yml): register-dataset ► data-prep ► model-training ► deploy-hosting ► update-main
```

## 1. Environment Setup
Install the libraries used across the pipeline. Versions are pinned so that training (local + GitHub Actions) and serving (Docker on HF Spaces) use **identical** `scikit-learn` / `xgboost` versions — otherwise the pickled model may fail to load.

In [ ]:
# Install pinned dependencies (quietly). Same pins are used in CI and in the Docker image.
%pip install -q pandas==2.2.2 numpy==1.26.4 scikit-learn==1.6.1 xgboost==2.1.4 huggingface_hub==0.35.3 joblib==1.4.2 matplotlib seaborn

In [ ]:
import os, sys, json, shutil, subprocess, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from huggingface_hub import HfApi, whoami

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

PY = sys.executable  # run the pipeline scripts with the same interpreter as this notebook
print("Python:", sys.version.split()[0])

**Hugging Face authentication.** The token is **never hard-coded**. Locally it is read from the `HF_TOKEN` environment variable
(or the cached `huggingface-cli login` token); in GitHub Actions it is injected from the repository secret `HF_TOKEN`.

In [ ]:
# Verify we are authenticated to the Hugging Face Hub (token comes from env / cached login, never printed)
hf_user = whoami(token=os.getenv("HF_TOKEN"))["name"]
print("Authenticated to Hugging Face as:", hf_user)

## 2. Data Registration
### 2.1 Create the master folder and the `data` sub-folder
`superkart_project/` is the master folder that holds every pipeline component:
```
superkart_project/
├── data/              raw + prepared data
├── model_building/    data_register.py, prep.py, train.py
├── deployment/        Dockerfile, app.py, requirements.txt, README.md
├── hosting/           hosting.py (pushes deployment files to the HF Space)
└── requirements.txt   dependencies for the GitHub Actions runners
```

In [ ]:
# Create the master folder and its sub-folders
MASTER = "superkart_project"
for sub in ["data", "model_building", "deployment", "hosting"]:
    os.makedirs(os.path.join(MASTER, sub), exist_ok=True)

# Place the raw dataset inside the data folder
if not os.path.exists(f"{MASTER}/data/SuperKart.csv"):
    shutil.copy("SuperKart.csv", f"{MASTER}/data/SuperKart.csv")
print(sorted(os.listdir(MASTER)), "|", os.listdir(f"{MASTER}/data"))

### 2.2 Quick look at the raw data (EDA)
A short exploratory pass to decide which cleaning steps the pipeline must perform.

In [ ]:
raw = pd.read_csv(f"{MASTER}/data/SuperKart.csv")
print("Shape:", raw.shape)
raw.head()

In [ ]:
raw.info()
print("\nMissing values per column:", raw.isna().sum().sum())
print("Duplicate rows:", raw.duplicated().sum())

In [ ]:
raw.describe().T.round(2)

In [ ]:
# Unique values of categorical columns -> spot inconsistent labels
for col in raw.select_dtypes("object").columns:
    print(f"{col:<26} nunique={raw[col].nunique():<5}",
          raw[col].unique()[:8] if raw[col].nunique() > 20 else raw[col].unique())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
sns.histplot(raw["Product_Store_Sales_Total"], kde=True, ax=axes[0]).set_title("Target distribution")
raw.groupby("Store_Id")["Product_Store_Sales_Total"].sum().div(1e6).plot.bar(ax=axes[1], color="steelblue")
axes[1].set_title("Total revenue by store (millions)"); axes[1].set_ylabel("Revenue (M)")
sns.heatmap(raw.select_dtypes("number").corr(), annot=True, fmt=".2f", cmap="Blues", ax=axes[2]).set_title("Correlation")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
sns.boxplot(data=raw, x="Store_Type", y="Product_Store_Sales_Total", ax=axes[0]).set_title("Sales by store type")
order = raw.groupby("Product_Type")["Product_Store_Sales_Total"].sum().sort_values(ascending=False).index
sns.barplot(data=raw, x="Product_Type", y="Product_Store_Sales_Total", estimator="sum", errorbar=None, order=order, ax=axes[1])
axes[1].set_title("Total revenue by product type"); axes[1].tick_params(axis="x", rotation=75)
plt.tight_layout(); plt.show()

# Each Store_Id maps to exactly one combination of store attributes
raw.groupby(["Store_Id", "Store_Establishment_Year", "Store_Size", "Store_Location_City_Type", "Store_Type"]).size().rename("rows")

**Observations**
- 8,763 rows × 12 columns, **no missing values and no duplicate rows**.
- `Product_Id` is unique for every row → pure identifier with no predictive value; it is dropped.
- `Store_Id` maps 1-to-1 onto (`Store_Establishment_Year`, `Store_Size`, `Store_Location_City_Type`, `Store_Type`) → redundant identifier; dropped so the model learns from interpretable store attributes.
- `Product_Sugar_Content` has an inconsistent label **`reg`** that must be merged into **`Regular`**.
- The target is roughly bell-shaped (mean ≈ 3,464). `Product_MRP` (r = 0.79) and `Product_Weight` (r = 0.74) are the strongest numeric drivers; `Product_Allocated_Area` is ~uncorrelated.
- **OUT004 (Supermarket Type 2, Tier 2)** generates by far the most revenue; **OUT002 (Food Mart, Tier 3)** the least; Departmental Stores have the highest per-product sales.
- Fruits & Vegetables and Snack Foods are the top revenue categories.

### 2.3 Register the raw data on the Hugging Face dataset space
`data_register.py` creates the dataset repo (if it does not already exist) and uploads `SuperKart.csv`.

In [ ]:
%%writefile superkart_project/model_building/data_register.py
"""Step 1 - Register the raw SuperKart dataset on the Hugging Face Hub (dataset repo)."""
import os
from huggingface_hub import HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError

REPO_ID = "shravanda/superkart-sales-data"
REPO_TYPE = "dataset"
RAW_FILE = "superkart_project/data/SuperKart.csv"

api = HfApi(token=os.getenv("HF_TOKEN"))  # token from env (GitHub secret) or cached login

# Create the dataset repo only if it does not exist yet (idempotent for CI re-runs)
try:
    api.repo_info(repo_id=REPO_ID, repo_type=REPO_TYPE)
    print(f"Dataset repo '{REPO_ID}' already exists - reusing it.")
except RepositoryNotFoundError:
    create_repo(repo_id=REPO_ID, repo_type=REPO_TYPE, private=False, token=os.getenv("HF_TOKEN"))
    print(f"Dataset repo '{REPO_ID}' created.")

# Upload the raw csv to the root of the dataset repo
api.upload_file(
    path_or_fileobj=RAW_FILE,
    path_in_repo="SuperKart.csv",
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    commit_message="Register raw SuperKart data",
)
print(f"Uploaded {RAW_FILE} -> https://huggingface.co/datasets/{REPO_ID}")

In [ ]:
!{PY} superkart_project/model_building/data_register.py

In [ ]:
# Confirm the file is now in the dataset repo
HfApi().list_repo_files("shravanda/superkart-sales-data", repo_type="dataset")

## 3. Data Preparation
`prep.py`:
1. Loads the dataset **directly from the Hugging Face dataset repo** (`hf://datasets/...`),
2. Cleans it — de-duplicates, standardises `reg → Regular`, drops the identifier columns `Product_Id` and `Store_Id`,
3. Splits it 80/20 into train/test (`random_state=42`) and saves the four files locally in `superkart_project/data/`,
4. Uploads the train/test files back to the same dataset repo.

In [ ]:
%%writefile superkart_project/model_building/prep.py
"""Step 2 - Load raw data from the HF Hub, clean it, split it and push the splits back to the Hub."""
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from huggingface_hub import HfApi

REPO_ID = "shravanda/superkart-sales-data"
DATASET_PATH = f"hf://datasets/{REPO_ID}/SuperKart.csv"
OUT_DIR = "superkart_project/data"
TARGET = "Product_Store_Sales_Total"

api = HfApi(token=os.getenv("HF_TOKEN"))

# ---- Load directly from the Hugging Face dataset repo
df = pd.read_csv(DATASET_PATH)
print("Loaded from HF Hub:", df.shape)

# ---- Cleaning
df = df.drop_duplicates()
# Harmonise the inconsistent sugar-content label ('reg' is the same as 'Regular')
df["Product_Sugar_Content"] = df["Product_Sugar_Content"].replace({"reg": "Regular"})
# Drop identifier columns: Product_Id is unique per row, Store_Id is fully described by the store attributes
df = df.drop(columns=["Product_Id", "Store_Id"])
df = df.dropna(subset=[TARGET])
print("After cleaning:", df.shape)
print("Sugar content levels:", sorted(df["Product_Sugar_Content"].unique()))

# ---- Split into features / target and train / test
X = df.drop(columns=[TARGET])
y = df[TARGET]
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=42)

# ---- Save locally
os.makedirs(OUT_DIR, exist_ok=True)
splits = {"Xtrain.csv": Xtrain, "Xtest.csv": Xtest, "ytrain.csv": ytrain, "ytest.csv": ytest}
for name, frame in splits.items():
    frame.to_csv(os.path.join(OUT_DIR, name), index=False)
print(f"Train: {Xtrain.shape}  Test: {Xtest.shape}")

# ---- Upload the splits back to the HF dataset repo
for name in splits:
    api.upload_file(
        path_or_fileobj=os.path.join(OUT_DIR, name),
        path_in_repo=name,
        repo_id=REPO_ID,
        repo_type="dataset",
        commit_message=f"Upload {name}",
    )
    print(f"Uploaded {name}")

In [ ]:
!{PY} superkart_project/model_building/prep.py

In [ ]:
# Files now present in the HF dataset repo
HfApi().list_repo_files("shravanda/superkart-sales-data", repo_type="dataset")

**Observations:** the cleaned data has 9 features + target. The 80/20 split gives **7,010 training rows and 1,753 test rows**;
all four split files are stored locally and versioned on the Hugging Face dataset repo, so training (locally or in CI) always reads the same data.

## 4. Model Training and Registration
`train.py`:
- **Loads train/test data from the Hugging Face dataset repo**,
- Builds a `Pipeline` = `ColumnTransformer` (StandardScaler for numeric, One-Hot for categorical) + regressor,
- **Defines two candidate models with parameter grids** — **Random Forest** and **XGBoost**,
- **Tunes** each with 5-fold `GridSearchCV` (scoring = RMSE),
- **Evaluates** each tuned model on train and test (RMSE, MAE, R², Adj. R², MAPE),
- Selects the best model by **cross-validated RMSE** (the test set is only used for reporting, avoiding selection leakage),
- **Registers the best model** (+ metrics and a model card) in the Hugging Face model hub.

In [ ]:
%%writefile superkart_project/model_building/train.py
"""Step 3 - Tune, evaluate and register the best sales-forecasting model on the HF model hub."""
import os
import json
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from huggingface_hub import HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError

DATA_REPO = "shravanda/superkart-sales-data"
MODEL_REPO = "shravanda/superkart-sales-model"
MODEL_FILE = "best_superkart_model_v1.joblib"
OUT_DIR = "superkart_project/model_building"

api = HfApi(token=os.getenv("HF_TOKEN"))

# ---- 1. Load the train / test splits from the HF dataset repo
base = f"hf://datasets/{DATA_REPO}"
Xtrain = pd.read_csv(f"{base}/Xtrain.csv")
Xtest = pd.read_csv(f"{base}/Xtest.csv")
ytrain = pd.read_csv(f"{base}/ytrain.csv").squeeze("columns")
ytest = pd.read_csv(f"{base}/ytest.csv").squeeze("columns")
print(f"Train: {Xtrain.shape}  Test: {Xtest.shape}")

# ---- 2. Preprocessing: scale numeric features, one-hot encode categorical features
numeric_features = ["Product_Weight", "Product_Allocated_Area", "Product_MRP", "Store_Establishment_Year"]
categorical_features = ["Product_Sugar_Content", "Product_Type", "Store_Size", "Store_Location_City_Type", "Store_Type"]
preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown="ignore"), categorical_features),
)

# ---- 3. Define candidate models and their hyper-parameter grids
candidates = {
    "RandomForest": (
        RandomForestRegressor(random_state=42, n_jobs=-1),
        {
            "model__n_estimators": [100, 200],
            "model__max_depth": [8, 12, None],
            "model__min_samples_leaf": [1, 4],
            "model__max_features": [0.5, 1.0],
        },
    ),
    "XGBoost": (
        xgb.XGBRegressor(objective="reg:squarederror", random_state=42, n_jobs=-1),
        {
            "model__n_estimators": [100, 200, 300],
            "model__max_depth": [3, 5, 7],
            "model__learning_rate": [0.05, 0.1],
            "model__subsample": [0.8, 1.0],
            "model__colsample_bytree": [0.8, 1.0],
        },
    ),
}


def evaluate(model, X, y):
    """Regression metrics used to judge forecast quality."""
    pred = model.predict(X)
    n, p = X.shape
    r2 = r2_score(y, pred)
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y, pred))),
        "MAE": float(mean_absolute_error(y, pred)),
        "R2": float(r2),
        "Adj_R2": float(1 - (1 - r2) * (n - 1) / (n - p - 1)),
        "MAPE": float(mean_absolute_percentage_error(y, pred)),
    }


# ---- 4. Tune every candidate with 5-fold cross-validation and evaluate it
cv = KFold(n_splits=5, shuffle=True, random_state=42)
results, fitted = {}, {}
for name, (estimator, grid) in candidates.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
    search = GridSearchCV(pipe, grid, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1)
    search.fit(Xtrain, ytrain)
    fitted[name] = search.best_estimator_
    results[name] = {
        "best_params": {k.replace("model__", ""): v for k, v in search.best_params_.items()},
        "cv_rmse": float(-search.best_score_),
        "train": evaluate(search.best_estimator_, Xtrain, ytrain),
        "test": evaluate(search.best_estimator_, Xtest, ytest),
    }
    print(f"\n=== {name} ===")
    print("Best params:", results[name]["best_params"])
    print(f"CV RMSE: {results[name]['cv_rmse']:.2f}")
    print(pd.DataFrame({"train": results[name]["train"], "test": results[name]["test"]}).round(4).T.to_string())

# ---- 5. Select the best model on cross-validated RMSE (test set is for reporting only)
best_name = min(results, key=lambda k: results[k]["cv_rmse"])
best_model = fitted[best_name]
print(f"\nBest model: {best_name} (CV RMSE = {results[best_name]['cv_rmse']:.2f}, "
      f"test R2 = {results[best_name]['test']['R2']:.4f})")

# ---- 6. Save the model + metrics locally
os.makedirs(OUT_DIR, exist_ok=True)
model_path = os.path.join(OUT_DIR, MODEL_FILE)
joblib.dump(best_model, model_path)
metrics = {"best_model": best_name, "results": results}
metrics_path = os.path.join(OUT_DIR, "model_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

# ---- 7. Register the best model in the HF model hub
try:
    api.repo_info(repo_id=MODEL_REPO, repo_type="model")
    print(f"Model repo '{MODEL_REPO}' already exists - reusing it.")
except RepositoryNotFoundError:
    create_repo(repo_id=MODEL_REPO, repo_type="model", private=False, token=os.getenv("HF_TOKEN"))
    print(f"Model repo '{MODEL_REPO}' created.")

t = results[best_name]["test"]
card = f"""---
license: mit
library_name: sklearn
tags: [tabular-regression, sales-forecasting, {best_name.lower()}]
datasets: [{DATA_REPO}]
---
# SuperKart Sales Forecasting Model

Predicts `Product_Store_Sales_Total` (revenue of a product in a store) for SuperKart.
Serialized scikit-learn `Pipeline` (preprocessing + **{best_name}**) tuned with 5-fold GridSearchCV.

| Metric (test set) | Value |
|---|---|
| RMSE | {t['RMSE']:.2f} |
| MAE | {t['MAE']:.2f} |
| R2 | {t['R2']:.4f} |
| MAPE | {t['MAPE']:.4f} |

Best hyper-parameters: `{results[best_name]['best_params']}`

```python
import joblib
from huggingface_hub import hf_hub_download
model = joblib.load(hf_hub_download("{MODEL_REPO}", "{MODEL_FILE}"))
```
"""
card_path = os.path.join(OUT_DIR, "MODEL_CARD.md")
with open(card_path, "w") as f:
    f.write(card)

for local, remote in [(model_path, MODEL_FILE), (metrics_path, "model_metrics.json"), (card_path, "README.md")]:
    api.upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=MODEL_REPO, repo_type="model",
                    commit_message=f"Register {best_name} model artefact: {remote}")
print(f"Registered best model -> https://huggingface.co/{MODEL_REPO}")

In [ ]:
%%time
!{PY} superkart_project/model_building/train.py

In [ ]:
# Side-by-side comparison of the tuned candidates
with open("superkart_project/model_building/model_metrics.json") as f:
    metrics = json.load(f)

rows = []
for name, r in metrics["results"].items():
    for split in ["train", "test"]:
        rows.append({"model": name, "split": split, "cv_rmse": r["cv_rmse"], **r[split]})
comparison = pd.DataFrame(rows).set_index(["model", "split"]).round(4)
print("Selected model:", metrics["best_model"])
comparison

In [ ]:
# Sanity check: download the registered model back from the hub and score the test set
import joblib
from huggingface_hub import hf_hub_download
from sklearn.metrics import r2_score

model = joblib.load(hf_hub_download("shravanda/superkart-sales-model", "best_superkart_model_v1.joblib", force_download=True))
Xtest = pd.read_csv("superkart_project/data/Xtest.csv")
ytest = pd.read_csv("superkart_project/data/ytest.csv").squeeze("columns")
pred = model.predict(Xtest)
print("R2 of model downloaded from the HF hub:", round(r2_score(ytest, pred), 4))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].scatter(ytest, pred, s=8, alpha=0.5)
axes[0].plot([ytest.min(), ytest.max()], [ytest.min(), ytest.max()], "r--")
axes[0].set(xlabel="Actual sales", ylabel="Predicted sales", title="Actual vs predicted (test)")

# Feature importance of the final estimator, mapped back to the encoded feature names
names = model.named_steps["preprocessor"].get_feature_names_out()
imp = pd.Series(model.named_steps["model"].feature_importances_, index=[n.split("__")[1] for n in names])
imp.sort_values().tail(12).plot.barh(ax=axes[1], color="steelblue", title="Top feature importances")
plt.tight_layout(); plt.show()

**Observations**
- Both tuned tree ensembles explain more than 90 % of the variance in product-store revenue on unseen data; train and test scores are close, so the selected model **generalises well with little over-fitting**.
- The model with the lowest **cross-validated RMSE** is registered on the HF model hub together with its metrics and a model card; the reload check above confirms the hub artefact gives identical results.
- **Product_MRP** and **Product_Weight** dominate the predictions, followed by store characteristics (store type / city tier) — consistent with the EDA.

## 5. Model Deployment
The frontend is a **Streamlit** app packaged as a **Docker** Space on Hugging Face.

### 5.1 Dockerfile

In [ ]:
%%writefile superkart_project/deployment/Dockerfile
# Lightweight Python base image (same Python minor version as training)
FROM python:3.11-slim

# Hugging Face Spaces run containers as user 1000 -> create it so the HF cache / streamlit config are writable
RUN useradd -m -u 1000 user
USER user
ENV HOME=/home/user \
    PATH=/home/user/.local/bin:$PATH \
    PYTHONUNBUFFERED=1 \
    STREAMLIT_BROWSER_GATHER_USAGE_STATS=false

WORKDIR $HOME/app

# Install dependencies first to leverage Docker layer caching
COPY --chown=user requirements.txt .
RUN pip install --no-cache-dir --upgrade pip && pip install --no-cache-dir -r requirements.txt

# Copy the application code
COPY --chown=user . .

# HF Spaces routes traffic to port 7860
EXPOSE 7860

# Launch the Streamlit app, listening on all interfaces
CMD ["streamlit", "run", "app.py", "--server.port=7860", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false"]

### 5.2 Dependencies file for the deployment

In [ ]:
%%writefile superkart_project/deployment/requirements.txt
pandas==2.2.2
numpy==1.26.4
scikit-learn==1.6.1
xgboost==2.1.4
joblib==1.4.2
huggingface_hub==0.35.3
streamlit==1.39.0

### 5.3 Streamlit app — loads the model from the HF model hub and turns the inputs into a DataFrame
- **Single prediction**: a form for one product–store combination; store presets pre-fill the store attributes of SuperKart's four outlets.
- **Batch prediction**: upload a CSV with the 9 feature columns to forecast many rows at once and download the results.

In [ ]:
%%writefile superkart_project/deployment/app.py
"""SuperKart Sales Forecast - Streamlit frontend served from a Hugging Face Docker Space."""
import joblib
import pandas as pd
import streamlit as st
from huggingface_hub import hf_hub_download

MODEL_REPO = "shravanda/superkart-sales-model"
MODEL_FILE = "best_superkart_model_v1.joblib"

FEATURES = ["Product_Weight", "Product_Sugar_Content", "Product_Allocated_Area", "Product_Type", "Product_MRP",
            "Store_Establishment_Year", "Store_Size", "Store_Location_City_Type", "Store_Type"]
PRODUCT_TYPES = ["Baking Goods", "Breads", "Breakfast", "Canned", "Dairy", "Frozen Foods", "Fruits and Vegetables",
                 "Hard Drinks", "Health and Hygiene", "Household", "Meat", "Others", "Seafood", "Snack Foods",
                 "Soft Drinks", "Starchy Foods"]
# Attributes of SuperKart's existing outlets (from the training data) to pre-fill the store section
STORES = {
    "OUT001": (1987, "High", "Tier 2", "Supermarket Type1"),
    "OUT002": (1998, "Small", "Tier 3", "Food Mart"),
    "OUT003": (1999, "Medium", "Tier 1", "Departmental Store"),
    "OUT004": (2009, "Medium", "Tier 2", "Supermarket Type2"),
}
SIZES, TIERS = ["Small", "Medium", "High"], ["Tier 1", "Tier 2", "Tier 3"]
STORE_TYPES = ["Departmental Store", "Food Mart", "Supermarket Type1", "Supermarket Type2"]


@st.cache_resource
def load_model():
    """Download the registered model from the HF model hub once and cache it."""
    return joblib.load(hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE))


st.set_page_config(page_title="SuperKart Sales Forecast", page_icon="🛒", layout="wide")
st.title("🛒 SuperKart Sales Forecast")
st.write("Predict the total revenue (**Product_Store_Sales_Total**) a product will generate in a store. "
         f"Model: [`{MODEL_REPO}`](https://huggingface.co/{MODEL_REPO})")
model = load_model()

single, batch = st.tabs(["Single prediction", "Batch prediction (CSV)"])

with single:
    left, right = st.columns(2)
    with left:
        st.subheader("Product details")
        weight = st.number_input("Product weight", min_value=1.0, max_value=30.0, value=12.66, step=0.1)
        sugar = st.selectbox("Sugar content", ["Low Sugar", "Regular", "No Sugar"])
        area = st.slider("Allocated display area (ratio)", 0.0, 0.3, 0.068, step=0.001, format="%.3f")
        ptype = st.selectbox("Product type", PRODUCT_TYPES, index=PRODUCT_TYPES.index("Frozen Foods"))
        mrp = st.number_input("Product MRP", min_value=1.0, max_value=500.0, value=147.0, step=1.0)
    with right:
        st.subheader("Store details")
        preset = st.selectbox("Store", list(STORES) + ["Custom / new store"], index=3)
        year, size, tier, stype = STORES.get(preset, (2005, "Medium", "Tier 2", "Supermarket Type1"))
        custom = preset not in STORES
        year = st.number_input("Establishment year", 1950, 2030, year, disabled=not custom)
        size = st.selectbox("Store size", SIZES, index=SIZES.index(size), disabled=not custom)
        tier = st.selectbox("City type", TIERS, index=TIERS.index(tier), disabled=not custom)
        stype = st.selectbox("Store type", STORE_TYPES, index=STORE_TYPES.index(stype), disabled=not custom)

    # Collect the inputs into a single-row DataFrame with the training column names
    input_df = pd.DataFrame([{
        "Product_Weight": weight, "Product_Sugar_Content": sugar, "Product_Allocated_Area": area,
        "Product_Type": ptype, "Product_MRP": mrp, "Store_Establishment_Year": int(year),
        "Store_Size": size, "Store_Location_City_Type": tier, "Store_Type": stype,
    }])
    st.dataframe(input_df, hide_index=True, use_container_width=True)

    if st.button("Predict sales", type="primary"):
        prediction = float(model.predict(input_df)[0])
        st.success(f"Forecast Product_Store_Sales_Total: **{prediction:,.2f}**")

with batch:
    st.write("Upload a CSV containing these columns: " + ", ".join(f"`{c}`" for c in FEATURES))
    uploaded = st.file_uploader("CSV file", type="csv")
    if uploaded is not None:
        data = pd.read_csv(uploaded)
        missing = [c for c in FEATURES if c not in data.columns]
        if missing:
            st.error(f"Missing columns: {missing}")
        else:
            data["Product_Sugar_Content"] = data["Product_Sugar_Content"].replace({"reg": "Regular"})
            data["Predicted_Sales"] = model.predict(data[FEATURES]).round(2)
            st.metric("Total forecast revenue", f"{data['Predicted_Sales'].sum():,.2f}")
            st.dataframe(data, use_container_width=True)
            st.download_button("Download predictions", data.to_csv(index=False), "superkart_predictions.csv")

### 5.4 Space configuration (README with YAML front-matter)
Tells Hugging Face that this Space uses the **Docker SDK** and exposes port 7860.

In [ ]:
%%writefile superkart_project/deployment/README.md
---
title: SuperKart Sales Forecast
emoji: 🛒
colorFrom: blue
colorTo: green
sdk: docker
app_port: 7860
pinned: false
license: mit
short_description: Streamlit app forecasting SuperKart product-store revenue
---

# SuperKart Sales Forecast

Streamlit frontend for the SuperKart sales-forecasting model. The model is loaded at start-up from the
Hugging Face model hub and predicts `Product_Store_Sales_Total` for a product in a store.

### 5.5 Hosting script — pushes every deployment file to the Hugging Face Space

In [ ]:
%%writefile superkart_project/hosting/hosting.py
"""Step 4 - Push the deployment folder (Dockerfile, app.py, requirements.txt, README.md) to the HF Space."""
import os
from huggingface_hub import HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError

SPACE_ID = "shravanda/SuperKart-Sales-Forecast"
api = HfApi(token=os.getenv("HF_TOKEN"))

# Create a public Docker Space if it does not exist yet
try:
    api.repo_info(repo_id=SPACE_ID, repo_type="space")
    print(f"Space '{SPACE_ID}' already exists - updating it.")
except RepositoryNotFoundError:
    create_repo(repo_id=SPACE_ID, repo_type="space", space_sdk="docker", private=False, token=os.getenv("HF_TOKEN"))
    print(f"Space '{SPACE_ID}' created.")

api.upload_folder(
    folder_path="superkart_project/deployment",
    repo_id=SPACE_ID,
    repo_type="space",
    path_in_repo="",
    commit_message="Deploy SuperKart Streamlit app",
)
print(f"Deployed -> https://huggingface.co/spaces/{SPACE_ID}")

In [ ]:
!{PY} superkart_project/hosting/hosting.py

## 6. Automated GitHub Actions Workflow (CI/CD)
### 6.1 Dependencies for the GitHub Actions runners

In [ ]:
%%writefile superkart_project/requirements.txt
pandas==2.2.2
numpy==1.26.4
scikit-learn==1.6.1
xgboost==2.1.4
joblib==1.4.2
huggingface_hub==0.35.3

### 6.2 `pipeline.yml` — one job per ML step
| Job | What it does | Depends on |
|---|---|---|
| `register-dataset` | uploads the raw CSV to the HF dataset repo | — |
| `data-prep` | loads from HF, cleans, splits, uploads splits | register-dataset |
| `model-training` | tunes RF + XGBoost, evaluates, registers the best model | data-prep |
| `deploy-hosting` | pushes Dockerfile / app / requirements to the HF Space | model-training |
| `update-main` | commits the latest model metrics back to the `main` branch | deploy-hosting |

The workflow triggers **automatically on every push to `main`** (and can be run manually via `workflow_dispatch`).
The final job pushes the refreshed `model_metrics.json` back to `main` with the built-in `GITHUB_TOKEN`
(commits made with this token do not re-trigger the workflow, so there is no loop). `HF_TOKEN` is a repository secret.

In [ ]:
os.makedirs(".github/workflows", exist_ok=True)

In [ ]:
%%writefile .github/workflows/pipeline.yml
name: SuperKart MLOps Pipeline

on:
  push:
    branches: [main]          # every code update pushed to main runs the full pipeline
    paths-ignore:
      - "**.md"
  workflow_dispatch:          # allow manual runs from the Actions tab

permissions:
  contents: write             # needed by the update-main job to push to main

env:
  PYTHON_VERSION: "3.11"

jobs:
  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
          cache: pip
      - name: Install dependencies
        run: pip install -r superkart_project/requirements.txt
      - name: Upload raw dataset to the Hugging Face Hub
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python superkart_project/model_building/data_register.py

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
          cache: pip
      - name: Install dependencies
        run: pip install -r superkart_project/requirements.txt
      - name: Clean, split and upload train/test data
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python superkart_project/model_building/prep.py

  model-training:
    needs: data-prep
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
          cache: pip
      - name: Install dependencies
        run: pip install -r superkart_project/requirements.txt
      - name: Tune, evaluate and register the best model
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python superkart_project/model_building/train.py
      - name: Keep the metrics for the update-main job
        uses: actions/upload-artifact@v4
        with:
          name: model-metrics
          path: superkart_project/model_building/model_metrics.json

  deploy-hosting:
    needs: model-training
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
          cache: pip
      - name: Install dependencies
        run: pip install -r superkart_project/requirements.txt
      - name: Push deployment files to the Hugging Face Space
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python superkart_project/hosting/hosting.py

  update-main:
    needs: deploy-hosting
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
        with:
          ref: main
      - uses: actions/download-artifact@v4
        with:
          name: model-metrics
          path: superkart_project/model_building
      - name: Commit refreshed model metrics to main
        run: |
          git config user.name  "github-actions[bot]"
          git config user.email "41898282+github-actions[bot]@users.noreply.github.com"
          git add superkart_project/model_building/model_metrics.json
          if git diff --cached --quiet; then
            echo "Metrics unchanged - nothing to push."
          else
            git commit -m "ci: update model metrics from run ${{ github.run_number }} [skip ci]"
            git push origin HEAD:main
          fi

### 6.3 Repository housekeeping
Generated artefacts (train/test splits, the serialized model) live on the Hugging Face Hub, so they are kept out of Git.

In [ ]:
%%writefile .gitignore
.venv/
__pycache__/
.ipynb_checkpoints/
.DS_Store
superkart_project/data/X*.csv
superkart_project/data/y*.csv
superkart_project/model_building/*.joblib
superkart_project/model_building/MODEL_CARD.md
screenshots/*.tmp

In [ ]:
%%writefile README.md
# SuperKart Sales Forecasting — MLOps Pipeline

End-to-end MLOps pipeline that forecasts `Product_Store_Sales_Total` for SuperKart and ships it as a Streamlit app.

| | |
|---|---|
| HF Space (frontend) | https://huggingface.co/spaces/shravanda/SuperKart-Sales-Forecast |
| HF Dataset | https://huggingface.co/datasets/shravanda/superkart-sales-data |
| HF Model | https://huggingface.co/shravanda/superkart-sales-model |

```
.github/workflows/pipeline.yml      CI/CD: register ► prep ► train ► deploy ► update-main
superkart_project/
├── data/SuperKart.csv              raw data
├── model_building/
│   ├── data_register.py            register raw data on the HF dataset hub
│   ├── prep.py                     clean + split, push splits to HF
│   ├── train.py                    GridSearchCV (RF, XGBoost), evaluate, register best model
│   └── model_metrics.json          latest metrics (auto-updated by CI)
├── deployment/                     Dockerfile, app.py, requirements.txt, README.md (HF Space)
├── hosting/hosting.py              push deployment files to the HF Space
└── requirements.txt                CI dependencies
SuperKart_MLOps_Pipeline.ipynb      project notebook
```

The workflow runs on every push to `main`. It needs a repository secret **`HF_TOKEN`** (Hugging Face write token).

### 6.4 Push all files to GitHub
The repository's `HF_TOKEN` secret was added once with `gh secret set HF_TOKEN` (Settings ▸ Secrets and variables ▸ Actions).
The cell below commits the pipeline files and pushes them to `main`, which **automatically triggers the workflow**.

In [ ]:
# Commit and push the pipeline to the main branch -> triggers the GitHub Actions workflow
REMOTE = "https://github.com/shravanambudkar/superkart-mlops.git"
def git(cmd):
    out = subprocess.run(f"git {cmd}", shell=True, capture_output=True, text=True)
    print(f"$ git {cmd}\n{(out.stdout + out.stderr).strip()}")

git("init -b main") if not os.path.isdir(".git") else None
git("remote get-url origin") if subprocess.run("git remote get-url origin", shell=True, capture_output=True).returncode == 0 \
    else git(f"remote add origin {REMOTE}")
git("add .github superkart_project .gitignore README.md")
git('commit -m "Update SuperKart MLOps pipeline" --quiet')
git("pull --rebase origin main --quiet")
git("push -u origin main")
git("log --oneline -5")

## 7. Output Evaluation
### 7.1 GitHub
**Repository:** [https://github.com/shravanambudkar/superkart-mlops](https://github.com/shravanambudkar/superkart-mlops) &nbsp;|&nbsp; **Workflow runs:** [https://github.com/shravanambudkar/superkart-mlops/actions](https://github.com/shravanambudkar/superkart-mlops/actions)

Folder structure of the repository (tracked files):

In [ ]:
# Render the tracked-file tree of the GitHub repository
files = subprocess.run("git ls-files", shell=True, capture_output=True, text=True).stdout.split()
tree = {}
for path in files:
    node = tree
    for part in path.split("/"):
        node = node.setdefault(part, {})

def show(node, prefix=""):
    items = sorted(node.items(), key=lambda kv: (not kv[1], kv[0]))
    for i, (name, child) in enumerate(items):
        last = i == len(items) - 1
        print(prefix + ("└── " if last else "├── ") + name + ("/" if child else ""))
        show(child, prefix + ("    " if last else "│   "))

print("superkart-mlops/")
show(tree)

In [ ]:
from IPython.display import Image, display, Markdown

def show_shot(path, caption):
    display(Markdown(f"**{caption}**"))
    display(Image(filename=path, width=1100)) if os.path.exists(path) else print(f"(missing {path})")

show_shot("screenshots/github_repo.png", "Screenshot – GitHub repository folder structure")

In [ ]:
show_shot("screenshots/github_workflow.png", "Screenshot – executed GitHub Actions workflow (all jobs succeeded)")

In [ ]:
# Latest workflow runs, queried live from the GitHub API
import urllib.request
runs = json.load(urllib.request.urlopen("https://api.github.com/repos/shravanambudkar/superkart-mlops/actions/runs?per_page=5"))["workflow_runs"]
pd.DataFrame([{"run": r["run_number"], "event": r["event"], "status": r["status"], "conclusion": r["conclusion"],
               "commit": r["head_commit"]["message"].splitlines()[0][:60], "created": r["created_at"]} for r in runs])

### 7.2 Streamlit app on Hugging Face Spaces
**Space:** [https://huggingface.co/spaces/shravanda/SuperKart-Sales-Forecast](https://huggingface.co/spaces/shravanda/SuperKart-Sales-Forecast)

In [ ]:
# Space runtime status, queried live from the Hugging Face API
print("Space stage:", HfApi().get_space_runtime("shravanda/SuperKart-Sales-Forecast").stage)
show_shot("screenshots/streamlit_app.png", "Screenshot – SuperKart Streamlit app on Hugging Face Spaces (single prediction)")

## 8. Conclusions & Business Insights
**Pipeline**
- A fully automated, reproducible pipeline: data versioned on the HF dataset hub, model versioned on the HF model hub, app served from a Docker HF Space, and every step orchestrated by GitHub Actions on each push to `main` — no manual intervention needed to refresh forecasts.
- Scripts are idempotent (repos are created only if missing), dependencies are pinned identically for training and serving, and secrets are handled via `HF_TOKEN` only.

**Model**
- The tuned tree ensemble explains > 90 % of the variation in product-store revenue on unseen data, with closely matching train/test metrics → reliable for planning.
- Revenue is driven mainly by **product price (MRP)** and **product weight**, then by **store type / city tier**.

**Recommendations for SuperKart**
- **Supply chain:** use the batch-prediction tab to forecast revenue for the full product catalogue per store and size procurement accordingly; prioritise high-MRP items in Supermarket Type 2 / Departmental stores.
- **Regional planning:** OUT004 (Supermarket Type 2, Tier 2) is the revenue engine; OUT002 (Food Mart, Tier 3) lags — target it with assortment and promotion changes, and use the "custom store" option to benchmark potential new outlets before opening them.
- **Display space:** allocated display area barely affects revenue → shelf space can be re-allocated toward high-value categories without hurting sales.
- **Next steps:** add time-stamped transactions to forecast weekly/monthly/quarterly revenue directly, add data-drift monitoring and a model-promotion gate (deploy only if test RMSE improves) to the workflow.